In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# 1. train.csv 데이터 로드
df_train = pd.read_csv('train.csv')

# =========================================================================
# 2. 데이터(X)와 타겟(y) 분리 및 설정
# =========================================================================
X_train = df_train.iloc[:, :-1]
y_train = df_train.iloc[:, -1]

"""
🚨 [시험장 돌발상황 대처용 주석]
만약 타겟 열이 맨 끝이 아니라 '중간'이나 '맨 앞'에 섞여서 출제된 경우, 
위의 iloc 라인을 주석 처리하고 아래 3줄의 주석을 해제하여 직접 지정하세요!

target_col_name = 'Class'  # <-- 시험지에 적힌 정확한 타겟 컬럼명 입력
X_train = df_train.drop(columns=[target_col_name], errors='ignore')
y_train = df_train[target_col_name]
"""
# =========================================================================

# 3. [파생변수 조합] 이름 몰라도 위치로 자동 조립
base_col_name = X_train.columns[-1] 
t1, t2 = X_train.columns[0], X_train.columns[1]

X_train['New_Div_Feature'] = X_train[t1] / (X_train[base_col_name] + 1e-3)
X_train['New_Weighted_Feature'] = (X_train[t1] * 2.0) + X_train[t2]

# 4. 종합 연관성 순위 계산 및 상위 5개 '위치(인덱스 번호)' 추출
corr_rank = X_train.corrwith(y_train).abs().rank(ascending=False)
rf_final = RandomForestClassifier(n_estimators=50, class_weight='balanced', n_jobs=-1, random_state=40)
rf_final.fit(X_train.values, y_train.values) # .values로 이름 지우기 [cite: 2]
rf_rank = pd.Series(rf_final.feature_importances_, index=X_train.columns).rank(ascending=False)

# 컬럼의 '위치 번호(숫자)'를 리스트로 추출 [cite: 2]
top_5_indices = (corr_rank + (rf_rank * 1.5)).sort_values().index[:5]
top_5_pos = [X_train.columns.get_loc(col) for col in top_5_indices]

# 5. 모델 학습 진행 (.values로 컬럼명 완전 제거) [cite: 3]
X_train_final = X_train[top_5_indices].values 

final_scaler = StandardScaler()
X_train_scaled = final_scaler.fit_transform(X_train_final)

# 임계값 조정 없이 자체 밸런싱이 완벽한 모델 구축
best_model = RandomForestClassifier(
    n_estimators=300,            # 트리 개수 300개로 확장 (일반화 성능 극대화) [cite: 14]
    criterion='entropy',         # 불균형 데이터 대응력 강화 [cite: 42]
    max_features='sqrt',         # 정예 5개 변수 시너지 최적화 [cite: 50]
    min_samples_split=5,         # 과적합 방지 [cite: 15]
    min_samples_leaf=2,          # 리프 노드 제한으로 처음 보는 데이터 방어력 리셋 [cite: 63, 64]
    class_weight='balanced',     # 소수 클래스 가중치 부여로 Macro F1 사수 [cite: 16, 42]
    n_jobs=-1, 
    random_state=40
)
best_model.fit(X_train_scaled, y_train.values) # ★ 핵심: 타겟 이름도 제거 [cite: 3]

# 6. 규칙 파일 보관
joblib.dump(best_model, 'best_rf_model.pkl')
joblib.dump(final_scaler, 'final_scaler.pkl')
joblib.dump(top_5_pos, 'top_5_pos.pkl')

print("=" * 60)
print("▶ [성공] 하이퍼파라미터가 안정화된 고성능 pkl 저장을 완료했습니다.")
print(f"▶ 선정된 열 위치 번호: {top_5_pos}")
print("=" * 60)

▶ [성공] 안정성이 강화된 최적 파라미터 pkl 저장을 마쳤습니다.
▶ 선정된 열 위치 번호: [7, 1, 11, 6, 3]


In [12]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import classification_report, f1_score

# =========================================================================
# 4. [조건 4] 테스트 데이터 실시간 구동 환경 선구축 (정석 복귀)
# =========================================================================
def predict_realtime_test_set(test_csv_path):
    # ① 실시간 테스트 데이터셋 로드 [cite: 300]
    df_test = pd.read_csv(test_csv_path)
    
    # 🚨 [시험장 대처] 타겟 컬럼이 포함되어 있다면 전처리 전 미리 제거
    if 'Class' in df_test.columns:
        df_test_features = df_test.drop(columns=['Class'])
    elif 'target' in df_test.columns:
        df_test_features = df_test.drop(columns=['target'])
    else:
        df_test_features = df_test.copy()

    # ② [파생변수 조합] 훈련과 동일 위치 기준 생성
    base_col_name = df_test_features.columns[-1]
    t1, t2 = df_test_features.columns[0], df_test_features.columns[1]
    
    df_test_features['New_Div_Feature'] = df_test_features[t1] / (df_test_features[base_col_name] + 1e-3)
    df_test_features['New_Weighted_Feature'] = (df_test_features[t1] * 2.0) + df_test_features[t2]
    
    # ③ 규칙 파일(학습 때 선정된 5개 열의 위치 번호) 로드
    loaded_pos = joblib.load('top_5_pos.pkl')
    
    # ④ 위치(인덱스 숫자)를 기준으로 5개 열만 필터링 후 컬럼명 즉시 박멸 (.values)
    X_realtime = df_test_features.iloc[:, loaded_pos].values
    
    # ⑤ 스케일러 및 최종 머신러닝 모델 파일 로드 [cite: 301]
    loaded_scaler = joblib.load('final_scaler.pkl')
    loaded_model = joblib.load('best_rf_model.pkl')
    
    # ⑥ 훈련 데이터와 동일한 기준으로 스케일링 전처리 수행 [cite: 301]
    X_realtime_scaled = loaded_scaler.transform(X_realtime)
    
    # ⑦ [정석 복귀] 불필요한 확률 추출을 제거하고 모델 본연의 예측 수행
    predictions = loaded_model.predict(X_realtime_scaled)
    
    return predictions


# =========================================================================
# 5. [조건 5] 실제 test.csv 가동 및 기말고사 성적 채점 [cite: 302]
# =========================================================================
test_file_name = 'test.csv' 

print("▶ [가동] 컬럼명 완전 박멸형 실시간 테스트 데이터 구동을 시작합니다...")
final_predictions = predict_realtime_test_set(test_file_name)


# =========================================================================
# [결과 검증] Macro Avg F1-Score 성적표 출력 [cite: 304]
# =========================================================================
print("\n" + "="*60)
print("★ [최종 성적표] 실시간 구동 모델 성능 채점 결과")
print("="*60)

df_actual = pd.read_csv(test_file_name)

actual_target_col = None
for col in ['Class', 'target', df_actual.columns[-1]]:
    if col in df_actual.columns:
        actual_target_col = col
        break

if actual_target_col:
    y_actual = df_actual[actual_target_col]
    print(classification_report(y_actual, final_predictions, digits=4))
    macro_f1 = f1_score(y_actual, final_predictions, average='macro')
    print("-"*60)
    print(f"🎯 교수님 제출용 기말고사 최종 평가 점수 (Macro F1): {macro_f1:.4f}")
    print("="*60)
else:
    print("▶ [안내] test.csv에 정답 열이 존재하지 않아 예측 라벨 출력으로 대체합니다.")
    print(f"▶ 최초 10개 예측 결과 샘플: {final_predictions[:10]}")

▶ [가동] 컬럼명 완전 박멸형 실시간 테스트 데이터 구동을 시작합니다...

★ [최종 성적표] 실시간 구동 모델 성능 채점 결과
              precision    recall  f1-score   support

           0     0.9775    0.9886    0.9831        88
           1     0.9091    0.8333    0.8696        12

    accuracy                         0.9700       100
   macro avg     0.9433    0.9110    0.9263       100
weighted avg     0.9693    0.9700    0.9694       100

------------------------------------------------------------
🎯 교수님 제출용 기말고사 최종 평가 점수 (Macro F1): 0.9263
